# GraphRAG vs Baseline Text-to-SQL — Evaluation Pipeline (Kaggle)

Notebook ini menjalankan **dua pipeline Text-to-SQL** pada Spider dev set, lalu mengevaluasinya dengan lapisan metrik skripsi. Semua logika ada di `src/` — notebook ini memanggilnya langkah demi langkah dan **menjelaskan apa yang terjadi di tiap langkah**, jadi bisa dibaca sebagai dokumentasi cara kerja pipeline.

## Dua pipeline yang dibandingkan

```
GraphRAG  (pipeline.py -> run_single)                         Baseline  (baseline.py -> run_single_baseline)
-------------------------------------------------------      -------------------------------------------------------
pertanyaan                                                    pertanyaan
  |                                                             |
[1] Stage 1: top-k TABEL (n-gram + cosine BGE-M3)             [1] Table-level linking: 1 embedding query vs embedding tabel
[2] Stage 2: top-k KOLOM di tabel kandidat                        |
[3] Graph traversal: shortest path antar kolom (JOIN)         [2] Table path tracing: shortest path antar TABEL via FK
[4] Schema context: DDL hanya kolom terpilih                  [3] Schema context: DDL SEMUA kolom tabel terpilih (tanpa pruning)
[5] Few-shot: top-k contoh dari train_spider.json             [4] Few-shot: TIDAK ada (zero-shot, hardcode di baseline.py)
[6] Prompt builder -> [7] Qwen2.5-Coder (4-bit, greedy) -> [8] SQL cleaner   (sama persis untuk keduanya)
```

## Alur notebook ini

| Bagian | Isi |
|---|---|
| 1-3 | Setup Kaggle, import, konfigurasi (`PipelineConfig` = sumber kebenaran tunggal) |
| 4-7 | Load data, bangun graph, load model, bangun index |
| 8 | Setup evaluasi (parser Spider, foreign-key map, gold schema) |
| 9-10 | **Walk-through 1 sampel**: tiap tahap pipeline diprint, biar kelihatan cara kerjanya |
| 11 | **Loop utama**: jalankan kedua pipeline atas semua sampel + evaluasi per query |
| 12-14 | Simpan `raw_logs`, ringkasan metrik, cross-check evaluator resmi Spider |
| 15 | **Analisis 6 dimensi** (QVT otomatis, Dimensi 1-6, laporan + JSON) lewat `run_dimension_analysis()` |

> **Catatan run final:** untuk skripsi, set `SAMPLE_RATIO = 1.0` (full 1034 query). Data QVT (`build_qvt_variations.py`) menolak `raw_logs` hasil subsampling, karena `query_id` = posisi baris di `dev.json`.

In [ ]:
# ============================================================
# 1. SETUP — clone repo, install dependency, siapkan evaluator Spider
# Sel ini aman dijalankan ulang (semua langkah dicek dulu).
# ============================================================
import os, subprocess, urllib.request
from pathlib import Path

REPO_URL = "https://github.com/william-darma-wijaya/thesis-2803.git"

def sh(cmd: str) -> None:
    print(f"$ {cmd}")
    subprocess.run(cmd, shell=True, check=True)

# 1) cwd HARUS root repo: pipeline memakai path relatif (outputs/, data/raw_logs/, external/spider_eval/)
if not Path("src").is_dir():
    if Path("../src").is_dir():          # notebook dijalankan dari notebooks/
        os.chdir("..")
    else:                                # Kaggle: /kaggle/working masih kosong
        sh(f"git clone {REPO_URL}")
        os.chdir("thesis-2803")
REPO_ROOT = Path.cwd()
print("Repo root:", REPO_ROOT)

# 2) dependency (torch sudah ada di Kaggle). nltk dibutuhkan process_sql.py milik Spider.
sh("pip install -q -U 'bitsandbytes>=0.46.1' sentence-transformers transformers networkx pandas tqdm nltk")

# 3) evaluator resmi Spider -> external/spider_eval/ (JANGAN diedit, hanya di-download apa adanya)
spider_dir = REPO_ROOT / "external" / "spider_eval"
spider_dir.mkdir(parents=True, exist_ok=True)
for fname in ("evaluation.py", "process_sql.py"):
    target = spider_dir / fname
    if not target.exists():
        url = f"https://raw.githubusercontent.com/taoyds/spider/master/{fname}"
        print("download", url)
        urllib.request.urlretrieve(url, target)

# 4) tokenizer data untuk parser SQL Spider
import nltk
for pkg in ("punkt", "punkt_tab"):
    try:
        nltk.download(pkg, quiet=True)
    except Exception as e:
        print(f"[WARN] nltk.download({pkg}) gagal: {e}")
print("Setup selesai.")

## 2. Import

Semua modul dipanggil sebagai `src.<paket>.<modul>`. Yang dipakai notebook ini:

| Modul | Peran |
|---|---|
| `src.core.schema` | parse `tables.json` -> DataFrame -> graph NetworkX (node = kolom) |
| `src.retrieval.retrieval` | **GraphRAG**: two-stage linking, graph traversal, schema context (DDL) |
| `src.retrieval.baseline` | **Baseline**: graph per tabel, table-level linking |
| `src.generation.few_shot` | index embedding train set + retrieval contoh |
| `src.generation.generation` | prompt builder, LLM 4-bit, SQL cleaner |
| `src.experiments.pipeline` | `run_single()` (satu pertanyaan, GraphRAG) + helper raw_logs, confound diagnostics, dan `run_dimension_analysis()` (jembatan ke 6 dimensi) |
| `src.metrics.*`, `src.utils.*` | **evaluasi**: ESM/EX/CM (`esm_ex_cm`), gold schema (`sla`), normalisasi JOIN |

Sebagian helper di `pipeline.py` berawalan `_` (mis. `_column_nodes_to_predicted_schema`). Itu sengaja dipakai ulang supaya notebook **tidak menduplikasi** logika dan tidak bisa drift dari script CLI.

In [ ]:
import sys, json, re
import numpy as np
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer

try:
    from tqdm.auto import tqdm
except ImportError:
    tqdm = lambda x, **kw: x

sys.path.insert(0, str(REPO_ROOT))   # supaya `import src...` resolve

from src.core.config import PipelineConfig
from src.core.schema import load_spider_schema, build_schema_graph

from src.retrieval.retrieval import (
    build_schema_index, retrieve_candidate_tables, semantic_schema_linking,
    trace_schema_paths, build_schema_context, evaluate_schema_linking,
)
from src.retrieval.baseline import (
    BaselineResult, build_table_graph, build_table_index,
    semantic_linking_table_level, trace_table_paths, build_table_schema_context,
    evaluate_table_linking, run_single_baseline,
)
from src.generation.few_shot import build_few_shot_index, retrieve_few_shot_examples, format_few_shot_block
from src.generation.generation import load_model_and_tokenizer, build_prompt, generate_sql_with_token_count

from src.experiments.pipeline import (
    run_single, set_seed, PipelineResult, build_gold_schema_context,
    print_schema_linking_summary, run_dimension_analysis,
    _column_nodes_to_predicted_schema, _table_nodes_to_predicted_schema,
    _new_confound_diag, _update_confound_diag, _format_confound_diag,
)
from src.experiments.sweep import run_sweep_and_get_best

from src.metrics import esm_ex_cm
from src.metrics.sla import extract_ground_truth_schema
from src.utils.schema_utils import load_db_schema
from src.utils.sql_normalize import normalize_sql_file_for_parsing

print("Imports OK | CUDA available:", torch.cuda.is_available())

## 3. Konfigurasi

`PipelineConfig` (`src/core/config.py`) adalah **sumber kebenaran tunggal** — jangan hardcode nilai di tempat lain. Yang bisa diatur di notebook ini:

- `SAMPLE_RATIO` — fraksi dev set. `0.05` = trial cepat (~51 query). **`1.0` = run final** untuk skripsi.
- `RUN_SWEEP` — kalau `True`, jalankan `sweep.py` (tanpa LLM, cepat) untuk memilih `top_k_tables`/`top_k_columns`; kalau `False`, pakai nilai di `config.py` (masih **TBD** sampai sweep dijalankan).

Parameter penting: `top_k_tables=3`, `top_k_columns=5`, `few_shot_k=3` (hanya dipakai GraphRAG), `token_output_weight=3.0` (α di T = T_in + α·T_out, fixed dari proposal), `temperature=0.0` (greedy, deterministik), `seed=42`.

**Output** ditulis relatif ke root repo. Untuk run trial (`SAMPLE_RATIO < 1.0`) semuanya diarahkan ke `outputs/trial/` supaya **tidak menimpa** `data/raw_logs/` hasil run final.

In [ ]:
SAMPLE_RATIO = 0.05      # 0.05 = trial cepat | 1.0 = FULL dev set (wajib untuk QVT & run final skripsi)
RUN_SWEEP    = False     # True -> jalankan sweep top_k dulu (lihat sel berikutnya)

cfg = PipelineConfig()
# Kalau bukan di Kaggle, timpa lokasi data Spider, mis.:
# cfg = PipelineConfig(data_path=Path("data/spider_data"))

assert cfg.data_path.exists(), (
    f"Data Spider tidak ditemukan di {cfg.data_path}. Di Kaggle, tambahkan dataset Spider "
    "lewat 'Add Input', atau isi PipelineConfig(data_path=...) dengan lokasi yang benar."
)

IS_FULL = SAMPLE_RATIO >= 1.0
ARTIFACT_DIR = Path("outputs") if IS_FULL else Path("outputs/trial")
RAW_LOGS_DIR = Path("data/raw_logs") if IS_FULL else ARTIFACT_DIR / "raw_logs"
for d in (ARTIFACT_DIR / "predictions", ARTIFACT_DIR / "tables", RAW_LOGS_DIR):
    d.mkdir(parents=True, exist_ok=True)

set_seed(cfg.seed)   # random, numpy, torch, cudnn deterministik -> run reprodusibel

print(f"Data path       : {cfg.data_path}")
print(f"Embedding model : {cfg.embedding_model}")
print(f"LLM model       : {cfg.llm_model}")
print(f"top_k_tables    : {cfg.top_k_tables}   top_k_columns: {cfg.top_k_columns}")
print(f"few_shot_k      : {cfg.few_shot_k}   (GraphRAG saja; Baseline zero-shot)")
print(f"alpha (T_out)   : {cfg.token_output_weight}")
print(f"SAMPLE_RATIO    : {SAMPLE_RATIO}  ->  {'FULL RUN' if IS_FULL else 'TRIAL (output ke outputs/trial/)'}")
print(f"raw_logs ke     : {RAW_LOGS_DIR}")

### 3b. (Opsional) Hyperparameter sweep

`sweep.py` mencoba kombinasi `top_k_tables x top_k_columns` **tanpa LLM** (hanya embedding lookup) dan mengembalikan kombinasi dengan schema-linking terbaik. Set `RUN_SWEEP = True` di sel konfigurasi kalau mau menentukan nilai final; hasilnya langsung menimpa `cfg` untuk sisa notebook ini. Nilai final juga perlu dicatat manual ke `config.py` supaya run berikutnya konsisten.

In [ ]:
SWEEP_SAMPLE = 0.2
if RUN_SWEEP:
    best_tables, best_cols = run_sweep_and_get_best(sample_ratio=SWEEP_SAMPLE, cfg=cfg)
    cfg.top_k_tables, cfg.top_k_columns = best_tables, best_cols
    print(f"Sweep selesai -> top_k_tables={best_tables}, top_k_columns={best_cols}")
else:
    print(f"Sweep dilewati -> pakai config.py: top_k_tables={cfg.top_k_tables}, top_k_columns={cfg.top_k_columns}")

## 4. Load Data (Spider Dev Set)

`dev.json` = 1034 pasang (question, gold SQL, db_id). Subsampling (kalau `SAMPLE_RATIO < 1.0`) **identik dengan `run_comparison()`**: pakai `np.random.default_rng(cfg.seed)` sehingga sampel yang sama dipakai kedua pipeline dan bisa direproduksi.

Gold SQL subset ditulis ke `gold_subset.sql` dalam format resmi Spider (`SQL<TAB>db_id`) — dipakai nanti untuk cross-check dengan `evaluation.py` resmi.

> `query_id` di raw_logs = `dev_{posisi+1:04d}` **dalam sampel ini**. Hanya pada run penuh posisi itu sama dengan posisi di `dev.json` (syarat QVT).

In [ ]:
with open(cfg.dev_json, encoding="utf-8") as f:
    dev_data = json.load(f)

if SAMPLE_RATIO < 1.0:
    n = max(1, int(len(dev_data) * SAMPLE_RATIO))
    rng = np.random.default_rng(cfg.seed)
    indices = rng.choice(len(dev_data), size=n, replace=False)
    dev_subset = [dev_data[i] for i in sorted(indices)]
else:
    dev_subset = dev_data

gold_path = ARTIFACT_DIR / "gold_subset.sql"
with open(gold_path, "w", encoding="utf-8") as f:
    for item in dev_subset:
        f.write(item["query"].strip() + "\t" + item["db_id"] + "\n")

print(f"Total dev set : {len(dev_data)}")
print(f"Dievaluasi    : {len(dev_subset)} ({len(dev_subset)/len(dev_data)*100:.1f}%)")
print(f"Gold subset   : {gold_path}")
ex = dev_subset[0]
print("\nContoh sampel pertama:")
print(f"  db_id    : {ex['db_id']}")
print(f"  question : {ex['question']}")
print(f"  gold SQL : {ex['query']}")

## 5. Bangun Schema Graph

Dua graph dibangun dari `tables.json` dengan granularitas berbeda — **ini inti perbedaan struktural GraphRAG vs Baseline**:

**GraphRAG** — node per **kolom** (`{db}.{table}.{col}`), diproses `build_schema_graph()`:
- edge `belongs_to_pk`: kolom non-PK -> PK tabel yang sama
- edge `foreign_key`: kolom FK -> kolom PK tabel lain
- memungkinkan seleksi sampai level kolom

**Baseline** — node per **tabel** (`{db}.{table}`), diproses `build_table_graph()`:
- edge `foreign_key` antar tabel
- retrieval berhenti di level tabel; semua kolom tabel terpilih masuk prompt

In [ ]:
print("Loading schema from tables.json ...")
schema_df = load_spider_schema(cfg.tables_json)
print(f"Schema DataFrame: {len(schema_df)} baris")

col_graph = build_schema_graph(schema_df)          # GraphRAG: node = kolom
print(f"GraphRAG graph  : {col_graph.number_of_nodes():,} nodes | {col_graph.number_of_edges():,} edges")

tbl_graph = build_table_graph(schema_df)           # Baseline: node = tabel
print(f"Baseline graph  : {tbl_graph.number_of_nodes():,} nodes | {tbl_graph.number_of_edges():,} edges")

## 6. Load Model

Dua model di-load **sekali** dan dipakai bersama kedua pipeline (jadi perbedaan hasil murni dari retrieval, bukan model):

| Model | Kegunaan | Catatan |
|---|---|---|
| `BAAI/bge-m3` | schema linking **dan** few-shot retrieval | Jangan ganti salah satu tanpa ganti keduanya |
| `Qwen/Qwen2.5-Coder-7B-Instruct` | SQL generation | 4-bit NF4, greedy decode (`temperature=0`) |

Loading LLM butuh beberapa menit dan ~6 GB VRAM — pastikan Accelerator = GPU (T4).

In [ ]:
print(f"Loading embedding model: {cfg.embedding_model}")
embed_model = SentenceTransformer(cfg.embedding_model)

print(f"Loading LLM: {cfg.llm_model} (4-bit) ...")
llm, tokenizer = load_model_and_tokenizer(cfg)
print("Model siap.")

## 7. Bangun Index

Semua embedding disimpan **di memori** (tanpa vector DB eksternal) dan dibangun **sekali** di awal supaya tidak dihitung ulang di dalam loop:

- **Schema index GraphRAG** (`SchemaIndex`, per `db_id`): embedding tiap tabel + tiap kolom
- **Table index Baseline** (`TableSchemaIndex`, per `db_id`): embedding tiap tabel berformat `"table <nama> columns <kol1> <kol2> ..."`
- **Few-shot index** (`FewShotIndex`): embedding seluruh `train_spider.json` (~7000 pertanyaan). Contoh **hanya diambil dari train set**, bukan dev/test -> bukan kebocoran data (Spider cross-domain: DB train dan dev tidak beririsan).

In [ ]:
unique_db_ids = list(dict.fromkeys(item["db_id"] for item in dev_subset))
print(f"{len(unique_db_ids)} database unik di sampel ini")

col_cache = {db_id: build_schema_index(col_graph, db_id, embed_model)
             for db_id in tqdm(unique_db_ids, desc="Index kolom (GraphRAG)")}
tbl_cache = {db_id: build_table_index(tbl_graph, db_id, embed_model)
             for db_id in tqdm(unique_db_ids, desc="Index tabel (Baseline)")}

few_shot_idx = None
if cfg.few_shot_k > 0:
    print("Building few-shot index dari train set ...")
    few_shot_idx = build_few_shot_index(cfg.train_json, embed_model)
    print(f"Few-shot index: {len(few_shot_idx.examples)} contoh training")
else:
    print("few_shot_k=0 -> zero-shot untuk GraphRAG juga")

## 8. Setup Evaluasi

Sebelum menjalankan pipeline, siapkan alat evaluasinya. Ada **dua jalur evaluasi** dan penting untuk tahu bedanya:

| Jalur | Dipakai untuk | Kode |
|---|---|---|
| **In-process** (sumber angka utama) | ESM/EX/CM per query -> `raw_logs` -> 6 dimensi | `src/metrics/esm_ex_cm.py` (**custom**, membungkus parser & evaluator **resmi Spider**) |
| **Subprocess** (cross-check) | ESM/EX agregat | `external/spider_eval/evaluation.py` (**resmi Spider**, tidak diubah) |

Yang disiapkan sekali di sini (bukan per query, karena mahal):
- `db_schema` — schema per DB untuk menurunkan **gold schema** (kolom yang dibutuhkan gold SQL) -> dasar Schema Linking Accuracy (SLA)
- `kmaps` — peta foreign key per DB, dibutuhkan parser Spider
- `spider_schema_cache` — objek `Schema` parser Spider (butuh koneksi SQLite), dibuat lazy per `db_id`

**Definisi metrik singkat:**
- **ESM** — struktur SQL (parse tree) sama dengan gold. Di Spider, ESM ≡ EM.
- **EX** — hasil eksekusi di SQLite sama dengan gold (pakai `eval_exec_match()` resmi, order-sensitive).
- **CM** — kecocokan per klausa (SELECT, FROM, WHERE, GROUP BY, ORDER BY, ...), dipakai Dimensi 3.
- **Token Consumption** — T = T_in + α·T_out (α = 3.0).

In [ ]:
print("Loading db_schema + foreign-key maps ...")
db_schema = load_db_schema(str(cfg.tables_json))
kmaps = esm_ex_cm.build_kmaps(str(cfg.tables_json))
spider_schema_cache = {}

def db_path_of(db_id: str) -> str:
    return str(cfg.db_dir / db_id / f"{db_id}.sqlite")

def evaluate_pred(pred_sql: str, gold_sql: str, db_id: str):
    """Bungkus esm_ex_cm.evaluate_single_query(): 1 prediksi -> objek hasil (esm, ex, cm_per_clause, difficulty)."""
    path = db_path_of(db_id)
    if db_id not in spider_schema_cache:
        spider_schema_cache[db_id] = esm_ex_cm.build_schema_for_db(path)
    return esm_ex_cm.evaluate_single_query(pred_sql, gold_sql, db_id, path,
                                           spider_schema_cache[db_id], kmaps[db_id])

print(f"db_schema: {len(db_schema)} DB | kmaps: {len(kmaps)} DB")

## 9. Walk-through Satu Sampel: GraphRAG, Tahap demi Tahap

Sebelum loop penuh, kita bedah **satu pertanyaan** supaya jelas apa yang dikerjakan `run_single()`. Setiap sel di bawah = satu tahap pipeline. (Hasil di sini hanya demonstrasi dan **tidak dicatat** ke hasil akhir.)

Ganti `WALK_IDX` untuk melihat pertanyaan lain.

In [ ]:
WALK_IDX = 0
item = dev_subset[WALK_IDX]
q, gold, db = item["question"], item["query"], item["db_id"]
print(f"DB       : {db}")
print(f"Question : {q}")
print(f"Gold SQL : {gold}")

# ---- [1] STAGE 1: pilih top-k TABEL kandidat -----------------------------
# Pertanyaan dipecah jadi n-gram (uni/bi/trigram), tiap n-gram di-embed dengan BGE-M3,
# lalu dibandingkan (cosine) ke embedding tiap tabel. Skor tabel = skor n-gram terbaiknya (max-pool).
cand_tables = retrieve_candidate_tables(q, col_cache[db], embed_model, cfg)
print(f"\n[Stage 1] top-{cfg.top_k_tables} tabel kandidat: {cand_tables}")

In [ ]:
# ---- [2] STAGE 2: pilih top-k KOLOM di dalam tabel kandidat --------------
# semantic_schema_linking = Stage 1 + Stage 2 sekaligus; hasilnya nama kolom terpilih
# (top_k_columns terbaik per n-gram; kalau top_k_columns=0 dipakai threshold similarity).
detected_cols = semantic_schema_linking(
    col_graph, db, q,
    all_columns=col_cache[db].columns,
    col_embeddings=col_cache[db].col_embeddings,
    embed_model=embed_model, cfg=cfg, index=col_cache[db],
)
print(f"[Stage 2] kolom terdeteksi ({len(detected_cols)}): {detected_cols}")

# ---- [3] GRAPH TRAVERSAL: cari jalur JOIN antar kolom terpilih -----------
# Untuk tiap pasang kolom terpilih dicari shortest path di graph skema (lewat edge PK/FK).
# Node perantara di jalur itu (mis. kolom FK/PK penghubung tabel) ikut dimasukkan supaya LLM bisa menulis JOIN.
c_nodes, paths, relations = trace_schema_paths(col_graph, db, detected_cols)
column_nodes = list(set(c_nodes + [node for p in paths for node in p]))
print(f"\n[Traversal] {len(c_nodes)} node anchor, {len(paths)} jalur, {len(relations)} relasi")
for p in paths[:5]:
    print("   ", " -> ".join(n.split(".", 1)[1] for n in p))
print(f"[Traversal] total node yang masuk konteks: {len(column_nodes)}")
print("(Catatan: tidak ada langkah pruning tambahan — node jalur dipakai apa adanya, sama seperti run_single().)")

In [ ]:
# ---- [4] SCHEMA CONTEXT: bangun DDL dari kolom terpilih -------------------
schema_context = build_schema_context(col_graph, column_nodes)
print("[Schema context yang dikirim ke LLM]")
print(schema_context)

# Evaluasi retrieval (proxy internal, BUKAN SLA resmi — SLA dihitung nanti dari raw_logs via sla.py)
recall, precision = evaluate_schema_linking(gold, column_nodes, col_graph, db)
print(f"\nProxy schema-linking: recall={recall:.2f}  precision={precision:.2f}")

# ---- [5] FEW-SHOT: contoh mirip dari train_spider.json --------------------
few_shot_block = ""
if few_shot_idx is not None and cfg.few_shot_k > 0:
    examples = retrieve_few_shot_examples(q, few_shot_idx, embed_model, cfg)
    few_shot_block = format_few_shot_block(examples)
print("\n[Few-shot block]")
print(few_shot_block if few_shot_block else "(kosong: few_shot_k=0)")

In [ ]:
# ---- [6] PROMPT BUILDER ---------------------------------------------------
# Format hybrid: Task / Database Schema / Examples / Question / Instructions / Answer.
# Angka & string literal yang muncul di pertanyaan diekstrak dan disertakan supaya LLM memakai nilai persis.
extracted_values = {
    "strings": re.findall(r"'([^']*)'", q),
    "numbers": re.findall(r"\d+", q),
}
prompt = build_prompt(q, schema_context, extracted_values, few_shot_block)
n_in = len(tokenizer.encode(prompt))
print(prompt)
print(f"\n[T_in] prompt = {n_in} token")

# ---- [7] LLM + [8] SQL CLEANER --------------------------------------------
# Qwen2.5-Coder 4-bit, greedy. generate_sql_with_token_count() juga menghitung T_out
# dari output MENTAH sebelum dibersihkan, lalu SQL cleaner membuang markdown/alias aneh.
walk_pred, n_out = generate_sql_with_token_count(prompt, llm, tokenizer, cfg)
print(f"\n[T_out] {n_out} token   ->   T = T_in + {cfg.token_output_weight} x T_out = {n_in + cfg.token_output_weight * n_out:.0f}")
print(f"Pred SQL : {walk_pred}")
print(f"Gold SQL : {gold}")

### 9b. Evaluasi sampel yang sama (in-process)

`evaluate_single_query()` menjalankan tiga hal sekaligus atas pasangan (prediksi, gold): **ESM** (bandingkan parse tree), **EX** (eksekusi kedua SQL di SQLite), **CM per klausa**, plus **difficulty** yang diturunkan dari gold SQL (easy/medium/hard/extra, pakai `eval_hardness()` resmi Spider).

In [ ]:
walk_eval = evaluate_pred(walk_pred, gold, db)
walk_gold_schema = sorted(extract_ground_truth_schema(gold, db, db_schema))
print(f"difficulty    : {walk_eval.difficulty}")
print(f"ESM           : {walk_eval.esm}")
print(f"EX            : {walk_eval.ex}")
print(f"CM per klausa : {walk_eval.cm_per_clause}")
print(f"Gold schema   : {walk_gold_schema}   <- kolom yang dibutuhkan gold SQL (dasar SLA)")
print(f"Pred schema   : {_column_nodes_to_predicted_schema(col_graph, column_nodes)}   <- kolom yang dikirim GraphRAG")

## 10. Walk-through Satu Sampel: Baseline

Pertanyaan yang sama, sekarang lewat Baseline. Bedanya ada di **granularitas**: yang dipilih adalah **tabel utuh**, dan **semua kolom** tabel itu masuk prompt. Baseline juga **zero-shot** (`few_shot_block=""` di-hardcode di `run_single_baseline()`).

In [ ]:
# ---- [1] Table-level linking: SATU embedding query vs embedding tiap tabel (bukan n-gram) ----
detected_tables = semantic_linking_table_level(tbl_cache[db], q, embed_model, top_k=cfg.top_k_tables)
print(f"[Linking] top-{cfg.top_k_tables} tabel: {detected_tables}")

# ---- [2] Table path tracing: tambahkan tabel perantara via FK bila ada gap konektivitas ----
table_nodes = trace_table_paths(tbl_graph, detected_tables)
print(f"[Path tracing] tabel final: {table_nodes}")

# ---- [3] Schema context: DDL dengan SEMUA kolom tiap tabel (tanpa pruning) ----
bl_context = build_table_schema_context(tbl_graph, table_nodes)
print("\n[Schema context yang dikirim ke LLM]")
print(bl_context)

bl_recall, bl_precision = evaluate_table_linking(gold, table_nodes, tbl_graph, db)
print(f"\nProxy schema-linking: recall={bl_recall:.2f}  precision={bl_precision:.2f}")

# ---- [4] Prompt (tanpa few-shot) -> LLM -> cleaner: sama dengan GraphRAG ----
bl_prompt = build_prompt(q, bl_context, extracted_values, few_shot_block="")
bl_n_in = len(tokenizer.encode(bl_prompt))
bl_pred, bl_n_out = generate_sql_with_token_count(bl_prompt, llm, tokenizer, cfg)
print(f"\nT_in={bl_n_in}  T_out={bl_n_out}   (GraphRAG: T_in={n_in}  T_out={n_out})")
print(f"Pred SQL : {bl_pred}")
bl_eval = evaluate_pred(bl_pred, gold, db)
print(f"ESM={bl_eval.esm}  EX={bl_eval.ex}  difficulty={bl_eval.difficulty}")

## 11. Loop Utama — Kedua Pipeline atas Semua Sampel

Loop ini **setara dengan `run_comparison()`** di `pipeline.py`, hanya ditulis eksplisit. Untuk tiap pertanyaan:

1. `run_single()` -> prediksi GraphRAG (+ recall/precision proxy, kolom terpilih, T_in, T_out)
2. `run_single_baseline()` -> prediksi Baseline (+ tabel terpilih, T_in, T_out)
3. Evaluasi in-process kedua prediksi (ESM/EX/CM/difficulty) + turunkan **gold schema**
4. Tambahkan satu entri ke `raw_logs` tiap kondisi (skema 12 field, lihat `context/EVALUATION_ANALYSIS_GUIDE.md` Bagian 2)
5. Update **confound diagnostics** (read-only; tidak mengubah nilai metrik apa pun)

**Penanganan error** (penting untuk run panjang di Kaggle) — dua `try/except` sengaja terpisah:
- gagal saat *generate* -> fallback `"SELECT 1"` dan loop lanjut (tidak ada satu error pun yang mematikan run berjam-jam)
- gagal saat *evaluasi* -> prediksi tetap tersimpan, hanya entri raw_log query itu yang dilewati; jumlahnya dilaporkan di akhir

In [ ]:
graphrag_results, baseline_results = [], []
graphrag_raw_logs, baseline_raw_logs = [], []
graphrag_preds, baseline_preds = [], []
confound_diag = _new_confound_diag()
gen_errors, eval_errors = [], []
W = 72

for i, item in enumerate(tqdm(dev_subset, desc="GraphRAG + Baseline")):
    db_id, question, gold_sql = item["db_id"], item["question"], item["query"]

    # ---- GraphRAG ----
    try:
        g_pred, g_recall, g_prec, g_schema, g_column_nodes, g_n_in, g_n_out = run_single(
            question, gold_sql, db_id, col_graph, embed_model, llm, tokenizer, cfg,
            schema_index=col_cache[db_id], few_shot_index=few_shot_idx,
        )
    except Exception as e:
        gen_errors.append(("graphrag", i, db_id, repr(e)))
        g_pred, g_recall, g_prec, g_schema = "SELECT 1", 0.0, 0.0, ""
        g_column_nodes, g_n_in, g_n_out = [], 0, 0

    # ---- Baseline ----
    try:
        b_pred, b_recall, b_prec, b_table_nodes, b_n_in, b_n_out = run_single_baseline(
            question, gold_sql, db_id, tbl_graph, embed_model, llm, tokenizer, cfg,
            table_index=tbl_cache[db_id],
        )
    except Exception as e:
        gen_errors.append(("baseline", i, db_id, repr(e)))
        b_pred, b_recall, b_prec = "SELECT 1", 0.0, 0.0
        b_table_nodes, b_n_in, b_n_out = [], 0, 0

    graphrag_results.append(PipelineResult(index=i + 1, db_id=db_id, question=question, gold_sql=gold_sql,
                                           pred_sql=g_pred, recall=g_recall, precision=g_prec))
    baseline_results.append(BaselineResult(index=i + 1, db_id=db_id, question=question, gold_sql=gold_sql,
                                           pred_sql=b_pred, recall=b_recall, precision=b_prec))
    graphrag_preds.append(g_pred.strip())
    baseline_preds.append(b_pred.strip())

    # ---- Evaluasi in-process + raw_logs (try/except TERPISAH dari generation) ----
    try:
        gold_schema = sorted(extract_ground_truth_schema(gold_sql, db_id, db_schema))
        g_eval = evaluate_pred(g_pred, gold_sql, db_id)
        b_eval = evaluate_pred(b_pred, gold_sql, db_id)

        # instrumentasi read-only (tidak mengubah nilai raw_logs di bawah)
        _update_confound_diag(confound_diag, db_path_of(db_id), gold_sql, g_pred, g_eval, b_pred, b_eval)

        difficulty = g_eval.difficulty          # properti gold_sql -> g_eval & b_eval selalu sama
        query_id = f"dev_{i + 1:04d}"
        graphrag_raw_logs.append({
            "query_id": query_id, "db_id": db_id, "difficulty": difficulty,
            "gold_sql": gold_sql, "predicted_sql": g_pred,
            "gold_schema": gold_schema,
            "predicted_schema": _column_nodes_to_predicted_schema(col_graph, g_column_nodes),
            "token_input": g_n_in, "token_output": g_n_out,
            "esm_result": g_eval.esm, "ex_result": g_eval.ex,
            "cm_per_clause": g_eval.cm_per_clause,
        })
        baseline_raw_logs.append({
            "query_id": query_id, "db_id": db_id, "difficulty": difficulty,
            "gold_sql": gold_sql, "predicted_sql": b_pred,
            "gold_schema": gold_schema,
            "predicted_schema": _table_nodes_to_predicted_schema(tbl_graph, b_table_nodes),
            "token_input": b_n_in, "token_output": b_n_out,
            "esm_result": b_eval.esm, "ex_result": b_eval.ex,
            "cm_per_clause": b_eval.cm_per_clause,
        })
    except Exception as e:
        eval_errors.append((i, db_id, repr(e)))

    if i == 0 or (i + 1) % 20 == 0 or (i + 1) == len(dev_subset):
        print("\n" + "=" * W)
        print(f"  [{i + 1:>4}/{len(dev_subset)}]  DB: {db_id}")
        print("=" * W)
        print(f"  Q              : {question}")
        print(f"  GOLD           : {gold_sql}")
        print(f"  [GraphRAG] pred: {g_pred}   (R={g_recall*100:.0f}% P={g_prec*100:.0f}%)")
        print(f"  [Baseline] pred: {b_pred}   (R={b_recall*100:.0f}% P={b_prec*100:.0f}%)")

print(f"\nSelesai: {len(graphrag_preds)} sampel | raw_logs: {len(graphrag_raw_logs)} entri")
print(f"Error generation: {len(gen_errors)} | Error evaluasi: {len(eval_errors)}")
for err in (gen_errors + eval_errors)[:10]:
    print("  ", err)

## 12. Simpan Artefak

- `predictions.txt` / `baseline_predictions.txt` — satu SQL per baris, urutan = urutan `gold_subset.sql`
- `raw_logs/{graphrag,baseline}_log.json` — **input untuk semua dimensi analisis**
- `comparison_report.txt` — ringkasan retrieval + blok CONFOUND DIAGNOSTICS

Ada pengecekan jumlah baris karena `evaluation.py` mencocokkan gold dan prediksi **berdasarkan nomor baris**; satu baris meleset = seluruh skor salah.

> Kalau ada query yang gagal dievaluasi, `raw_logs` lebih pendek dari sampel. Untuk run final sebaiknya **0 error** — investigasi dulu sebelum lanjut ke QVT/dimensi.

In [ ]:
g_pred_path = ARTIFACT_DIR / "predictions" / "predictions.txt"
b_pred_path = ARTIFACT_DIR / "predictions" / "baseline_predictions.txt"
g_pred_path.write_text("\n".join(graphrag_preds) + "\n", encoding="utf-8")
b_pred_path.write_text("\n".join(baseline_preds) + "\n", encoding="utf-8")

for label, path in (("GraphRAG", g_pred_path), ("Baseline", b_pred_path)):
    n_lines = len([l for l in path.read_text(encoding="utf-8").split("\n") if l.strip()])
    status = "OK" if n_lines == len(dev_subset) else "TIDAK SAMA — ada SQL multi-baris/kosong, skor Spider bisa meleset!"
    print(f"{label:<9}: {n_lines} baris vs {len(dev_subset)} sampel -> {status}")

with open(RAW_LOGS_DIR / "graphrag_log.json", "w", encoding="utf-8") as f:
    json.dump(graphrag_raw_logs, f, ensure_ascii=False, indent=2)
with open(RAW_LOGS_DIR / "baseline_log.json", "w", encoding="utf-8") as f:
    json.dump(baseline_raw_logs, f, ensure_ascii=False, indent=2)
print(f"\nraw_logs -> {RAW_LOGS_DIR}/ ({len(graphrag_raw_logs)} graphrag, {len(baseline_raw_logs)} baseline)")

g_r = np.mean([r.recall for r in graphrag_results]) * 100
g_p = np.mean([r.precision for r in graphrag_results]) * 100
b_r = np.mean([r.recall for r in baseline_results]) * 100
b_p = np.mean([r.precision for r in baseline_results]) * 100
report = "\n".join([
    "", "=" * 70,
    "  COMPARISON REPORT — GraphRAG (column/node) vs Baseline (table/node)", "=" * 70,
    f"  {'Metric':<26} {'GraphRAG':>10}  {'Baseline':>10}  {'Delta (G-B)':>12}", "-" * 70,
    f"  {'Recall (proxy)':<26} {g_r:>9.2f}%  {b_r:>9.2f}%  {g_r-b_r:>+11.2f}%",
    f"  {'Precision (proxy)':<26} {g_p:>9.2f}%  {b_p:>9.2f}%  {g_p-b_p:>+11.2f}%",
    "-" * 70,
    "  (proxy = name-matching internal; SLA resmi dihitung dari raw_logs via sla.py)",
    "=" * 70, "",
]) + _format_confound_diag(confound_diag)
print(report)
(ARTIFACT_DIR / "tables" / "comparison_report.txt").write_text(report, encoding="utf-8")

## 13. Ringkasan Hasil (dari raw_logs)

Angka di sini dihitung dari `raw_logs` — data yang sama persis yang dibaca 6 dimensi, jadi konsisten dengan analisis berikutnya.

- **ESM / EX** — rata-rata `esm_result` / `ex_result` (in-process, jalur utama)
- **T_in / T_out / T** — rata-rata token; `T = T_in + α·T_out`, α = `cfg.token_output_weight`
- Breakdown **per difficulty** untuk EX dan ESM (dasar Dimensi 2)

> Schema Recall/Precision (proxy) tidak ada di tabel ini dengan sengaja — itu bukan SLA. Yang dilaporkan di skripsi adalah SLA dari `src/metrics/sla.py`, yang dihitung di Dimensi 1.

In [ ]:
def summarize(raw_logs: list, label: str) -> dict:
    df = pd.DataFrame(raw_logs)
    T = df["token_input"] + cfg.token_output_weight * df["token_output"]
    return {
        "Pipeline": label, "n": len(df),
        "ESM": df["esm_result"].mean(), "EX": df["ex_result"].mean(),
        "T_in": df["token_input"].mean(), "T_out": df["token_output"].mean(), "T": T.mean(),
    }

summary = pd.DataFrame([summarize(graphrag_raw_logs, "GraphRAG"),
                        summarize(baseline_raw_logs, "Baseline")]).set_index("Pipeline")
display(summary.round(4))
if len(graphrag_raw_logs) != len(dev_subset):
    print(f"[!] raw_logs ({len(graphrag_raw_logs)}) < sampel ({len(dev_subset)}): ada query yang gagal dievaluasi.")

order = ["easy", "medium", "hard", "extra"]
for metric, col in (("EX", "ex_result"), ("ESM", "esm_result")):
    tbl = pd.DataFrame({
        "GraphRAG": pd.DataFrame(graphrag_raw_logs).groupby("difficulty")[col].mean(),
        "Baseline": pd.DataFrame(baseline_raw_logs).groupby("difficulty")[col].mean(),
        "n": pd.DataFrame(graphrag_raw_logs).groupby("difficulty")[col].size(),
    }).reindex(order)
    print(f"\n{metric} per difficulty:")
    display(tbl.round(4))

## 14. Cross-check dengan Evaluator Resmi Spider (subprocess)

Sebagai pembanding independen, panggil `external/spider_eval/evaluation.py` (resmi, tidak diubah) pada file prediksi. Satu hal penting: **parser resmi Spider hanya mengenali `JOIN` polos**, jadi `INNER/LEFT/RIGHT/FULL/CROSS JOIN` membuatnya crash. Karena itu file dinormalisasi dulu lewat `normalize_sql_file_for_parsing()` (`IMPLEMENTATION_DECISIONS.md` poin 11):

- `--etype match`: normalisasi **penuh** (ESM tidak pernah mengeksekusi SQL)
- `--etype exec`: normalisasi **hanya INNER/CROSS JOIN** (`execution_safe_only=True`), karena SQL benar-benar dieksekusi dan `LEFT/RIGHT/FULL JOIN` tidak boleh diam-diam berubah makna

Konsekuensinya, jalur subprocess ini tetap gagal parse untuk outer JOIN (**known limitation**); angka in-process di Bagian 13 tidak punya batasan itu. Jadi angka Bagian 13 = sumber utama, bagian ini hanya sanity check. Selisih kecil wajar.

In [ ]:
def run_spider_official(pred_path: Path, label: str) -> None:
    tmp = ARTIFACT_DIR / "spider_eval_tmp"
    tmp.mkdir(parents=True, exist_ok=True)
    stem = pred_path.stem

    inputs = {
        "match": (
            normalize_sql_file_for_parsing(gold_path, tmp / f"{stem}_match_gold.txt", has_db_id_suffix=True,  execution_safe_only=False),
            normalize_sql_file_for_parsing(pred_path, tmp / f"{stem}_match_pred.txt", has_db_id_suffix=False, execution_safe_only=False),
        ),
        "exec": (
            normalize_sql_file_for_parsing(gold_path, tmp / f"{stem}_exec_gold.txt", has_db_id_suffix=True,  execution_safe_only=True),
            normalize_sql_file_for_parsing(pred_path, tmp / f"{stem}_exec_pred.txt", has_db_id_suffix=False, execution_safe_only=True),
        ),
    }
    for etype, (g_file, p_file) in inputs.items():
        print("=" * 60)
        print(f"{label} — Spider resmi, --etype {etype}")
        print("=" * 60)
        r = subprocess.run(
            [sys.executable, "external/spider_eval/evaluation.py",
             "--gold", str(g_file), "--pred", str(p_file),
             "--db", str(cfg.db_dir), "--table", str(cfg.tables_json), "--etype", etype],
            capture_output=True, text=True,
        )
        print(r.stdout)
        if r.returncode != 0:
            print(f"[WARN] evaluation.py exit code {r.returncode}\n{r.stderr[-800:]}")

run_spider_official(g_pred_path, "GraphRAG")
run_spider_official(b_pred_path, "Baseline")

## 15. Analisis 6 Dimensi (integrasi metrics -> dimensions)

Sampai Bagian 13 kita punya angka mentah per query. **Dimensi** adalah level di atasnya: menggabungkan beberapa metrik untuk menjawab satu pertanyaan penelitian. Semuanya dijalankan lewat satu fungsi, `run_dimension_analysis()` dari `pipeline.py` — **fungsi yang sama** dipanggil otomatis di akhir `python src/experiments/pipeline.py --baseline`, jadi hasil notebook dan hasil script CLI tidak bisa berbeda.

| Dimensi | Pertanyaan yang dijawab | Metrik yang digabung |
|---|---|---|
| **1. Efisiensi** | Apakah GraphRAG lebih hemat token tanpa mengorbankan akurasi? (RM1, hasil utama skripsi) | SLA, EX/ESM, Token Consumption, TEP |
| **2. Struktur SQL** | Saat EX dan ESM bergerak, apa artinya? (mis. EX naik tapi ESM turun = SQL benar tapi ditulis beda) | ESM x EX, per difficulty |
| **3. Komponen SQL** | Klausa mana yang paling terdampak pruning konteks? | CM per klausa (SELECT/WHERE/GROUP BY/...) |
| **4. Robustness** | Apakah jawaban tetap konsisten saat pertanyaan diparafrase? | QVT (+ silang ESM) |
| **5. Bottleneck** | Kalau akurasi rendah, gagal di retrieval (tabel/kolom) atau di generation? | SLA table-level & column-level x EX |
| **6. Ablation few-shot** | Berapa k few-shot yang cukup (elbow point)? | EX per k, Token Consumption per k |

**Aturan penting (keputusan di `IMPLEMENTATION_DECISIONS.md` poin 21, 23, 24):**

- **Dimensi 4 hanya jalan pada run penuh** (`SAMPLE_RATIO = 1.0`). Data QVT dibangun otomatis dari paraphrase alami `dev.json` (~470 SQL, masing-masing punya 2 pertanyaan), di-join ke `raw_logs` lewat `query_id` = posisi baris di `dev.json`. Pada run trial dimensi ini dilewati dengan sengaja supaya tidak tercampur data QVT lama.
- **Dimensi 5 bersyarat**: hanya dijalankan kalau Dimensi 2 menemukan kuadran "EX rendah, ESM rendah" (`escalate_to_dimension_5`). `ALWAYS_DIM5 = True` menimpa aturan ini.
- **Dimensi 6 butuh `EX_PER_K`** dari ablation (`ablation.py` lalu evaluator Spider) yang belum tersimpan otomatis (open item D), jadi diisi manual, mis. `{0: 45.2, 1: 52.1, 3: 54.0, 5: 53.8}` (skala 0-100). Kalau `None`, Dimensi 6 di-skip dengan penjelasan.
- Dimensi yang prasyaratnya belum ada **di-skip dengan alasan**, bukan crash.

In [ ]:
EX_PER_K    = None    # Dimensi 6: {k: EX 0-100}, isi dari hasil ablation. None -> Dimensi 6 di-skip
ALWAYS_DIM5 = False   # True -> paksa Dimensi 5 tanpa menunggu trigger dari Dimensi 2

# Membangun data QVT (bila run penuh), menjalankan Dimensi 1-6, mencetak laporan,
# dan menulis dimensions_results.json + dimensions_report.txt ke ARTIFACT_DIR/tables/
dim_results = run_dimension_analysis(
    raw_logs_dir=RAW_LOGS_DIR,
    out_dir=ARTIFACT_DIR / "tables",
    dev_json=cfg.dev_json,
    full_dev=IS_FULL,
    ex_per_k=EX_PER_K,
    always_dim5=ALWAYS_DIM5,
)

### 15b. Ringkasan Status & Temuan Utama

Tabel status menunjukkan dimensi mana yang benar-benar jalan (`ok`) dan mana yang di-skip beserta alasannya. Di bawahnya, satu baris "headline" per dimensi yang sudah `ok`. Laporan lengkap (tabel per difficulty, per klausa, dst.) ada di output sel sebelumnya dan di `dimensions_report.txt`.

In [ ]:
DIM_NAMES = {1: "Efisiensi", 2: "Struktur SQL", 3: "Komponen SQL", 4: "Robustness (QVT)", 5: "Bottleneck", 6: "Ablation few-shot"}

rows = []
for n, name in DIM_NAMES.items():
    rec = dim_results.get(f"dimension_{n}")
    if rec is None:
        rows.append({"Dimensi": f"{n}. {name}", "Status": "tidak dijalankan", "Keterangan": "run trial: butuh run penuh" if n == 4 else ""})
    else:
        rows.append({"Dimensi": f"{n}. {name}", "Status": rec["status"],
                     "Keterangan": (rec.get("reason") or rec.get("error") or "")[:150]})
display(pd.DataFrame(rows).set_index("Dimensi"))

def headline(n: int, res: dict) -> str:
    if n == 2:
        c = res["classification"]
        return f"{c['combination']} (dEX={c['delta_ex']:+.1f} poin, dESM={c['delta_esm']:+.1f} poin), eskalasi ke Dimensi 5: {c['escalate_to_dimension_5']}"
    if n == 3:
        b = res["biggest_drop"]
        return f"klausa dengan penurunan terbesar: {b['clause']} ({b['delta']:+.1f} poin), penurunan nyata: {b['is_actual_decrease']}"
    if n == 4:
        return f"dQVT={res['delta_qvt']:+.3f} -> {res['interpretation']['category']}"
    if n == 5:
        return "; ".join(f"{k}: retrieval dan generation sejalan={v['levels_agree']}" for k, v in res.items())
    if n == 6:
        return f"k_final={res['k_final']}, pola: {res['pattern']}"
    return "lihat laporan lengkap"

print("TEMUAN UTAMA")
for n, name in DIM_NAMES.items():
    rec = dim_results.get(f"dimension_{n}")
    if rec and rec["status"] == "ok":
        try:
            print(f"  Dimensi {n} ({name}): {headline(n, rec['result'])}")
        except Exception as e:
            print(f"  Dimensi {n} ({name}): [format headline gagal: {e!r}] -> lihat dimensions_results.json")

### 15c. Menjalankan Ulang Hanya Lapisan Analisis

Kalau `raw_logs` sudah ada (mis. dari run Kaggle sebelumnya) dan yang berubah hanya konfigurasi analisis (mis. `EX_PER_K` baru dari ablation), **tidak perlu generate ulang** — cukup:

```bash
python src/experiments/run_all_dimensions.py --ex-per-k 0=45.2,1=52.1,3=54.0,5=53.8
```

atau ulangi hanya sel 15 di atas.

## Catatan Desain yang Perlu Disebut di Metodologi

- **Baseline zero-shot vs GraphRAG few-shot.** `run_single_baseline()` memanggil `build_prompt(..., few_shot_block="")`, sedangkan GraphRAG memakai `few_shot_k=3`. Selisih GraphRAG–Baseline di notebook ini karena itu **bercampur** dengan efek few-shot. Isolasi efek retrieval-nya lewat ablation (`ablation.py`, few-shot k ∈ {0,1,3,5} untuk kedua mode), atau jalankan notebook ini dengan `cfg.few_shot_k = 0` untuk perbandingan retrieval murni.
- **Tidak ada path pruning.** Pipeline saat ini memakai node hasil graph traversal apa adanya (tanpa langkah pruning tambahan).
- **Few-shot hanya dari `train_spider.json`.** Retrieval dari train set, bukan kebocoran dev/test.
- **Proxy vs SLA.** Recall/Precision yang diprint pipeline adalah proxy name-matching, bukan SLA; SLA resmi dari `src/metrics/sla.py`.
- **ESM ≡ EM di Spider; EX order-sensitive** (`eval_exec_match()` resmi apa adanya). Diagnostik confound di Bagian 12 mengukur dampaknya tanpa mengubah metrik.